# Tutorial to run benchmark examples, autodiff branch
This notebook demonstrates how to run the benchmark problems in the paper *A Latent-Space
Framework for Stress-Based Multi-Material Topology Optimization* on the `autodiff` branch.
On this branch the gradients of the objective and the constraints are computed by automatic
differentiation (PyTorch autograd) through [PyTO-AD](https://github.com/UW-ERSL/PyTO-AD),
instead of the hand-derived sensitivities on `main`. The examples, settings, schedules and
optimizer are the same.

The entire workflow is described from start to finish: the material database, its principal
components, the latent dimension that follows, the trained decoder, the problem
being solved, how the gradients are computed, a check of those gradients, the result,
and a comparison with `main`.

Select the kernel **Python (lsr-stress-mmto-ad)**.

**To see the comparison with `main` without running anything:** run SETUP, set `CHOICE` in
section 1, and go to section 11. It loads the runs saved in `comparison/reference`.

| | Domain | Materials | Latent | Objective | Constraints | Elements | Iterations |
|---|---|---|---|---|---|---|---|
| L-bracket | 2D benchmark against Banh et al. | 3 | 2D | p-norm stress | volume fraction per material | 12,825 | 300 |
| U-bracket | 2D, slot corner stress riser | 6 | 2D | p-norm failure factor | mass | 9,001 | 350 |
| Supported beam | 3D octant model | 8 | 2D | p-norm failure factor | mass | 25,748 | 350 |
| Airplane bracket | 3D half model | 20 | 3D | p-norm failure factor | mass and cost | 49,676 | 300 |

The reference runs in `comparison/reference` cover the L-bracket and the U-bracket; their run
times on both branches are in section 11. The airplane bracket needs a cost constraint, which
the autodiff model does not have yet.

## SETUP

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, ".")

from utils import project_paths, records, figures, tables, compare, gradcheck

ROOT = project_paths.bootstrap()

import numpy as np
import torch
import run_io
from ExamplesPureStructural import (MMTOExamplesPureStructural,
                                    getMMTOProblemPureStructural)
from materialEncoder import MaterialEncoder
from MainPureStructural import run_topopt, Z0InitMethod, BetaScheduleScheme

print(ROOT)

## 1. Choose an example

Set `CHOICE` to one of:

| `CHOICE` | example | problem |
|---|---|---|
| 1 | L-bracket | benchmark against Banh et al., volume fraction per material |
| 2 | U-bracket | six materials, mass constrained |
| 3 | beam | eight materials, octant model |
| 4 | Airplane bracket | twenty alloys, mass and cost constrained |

You may also modify the `relative_filter_radius` and the `material_filter_radius` parameters to change the topological and material feature sizes respectively.

In [ ]:
CHOICE = 1                      # 1, 2, 3 or 4

EXAMPLES = {1: "L-bracket", 2: "U-bracket", 3: "beam", 4: "Airplane bracket"}
EXAMPLE = EXAMPLES[CHOICE]

SETUP = {
    "L-bracket": dict(
        problem="LBracketTopLoad_Stress_BM1",
        database="BM1_3Mats_2Attr",
        relative_filter_radius=3.5, material_filter_radius=3.5,
        maxIterations=300, beta=BetaScheduleScheme.PAPER_BM1,
        note="benchmark against Banh et al., volume fraction per material"),
    "U-bracket": dict(
        problem="UBracket_Stress_FF",
        database="6Materials_AF_3Attr",
        relative_filter_radius=3.5, material_filter_radius=2.5,
        pnorm_exponent=16.0, maxIterations=350, beta=BetaScheduleScheme.PAPER,
        note="six materials, mass constrained"),
    "beam": dict(
        problem="Beam_Liao_Pareto_FF",
        database="Beam_8Mats_3Attr_v4_spectrum2D",
        relative_filter_radius=None, material_filter_radius=None,
        maxIterations=350, beta=BetaScheduleScheme.PAPER,
        use_constant_poissons_ratio=True,
        constant_poissons_ratio=0.37,
        note="eight materials, octant model, mid-front mass budget"),
    "Airplane bracket": dict(
        problem="AirplaneBracket_Stress",
        database="CustomAlloys20_3D",
        relative_filter_radius=None, material_filter_radius=None,
        maxIterations=300, beta=BetaScheduleScheme.PAPER,
        note="twenty alloys, mass and cost constrained"),
}

cfg = SETUP[EXAMPLE]
problem = MMTOExamplesPureStructural[cfg["problem"]]
xlsx = f"./1-PureStructural/MaterialDataPureStructural/{cfg['database']}.xlsx"

print(f"{EXAMPLE}: {cfg['note']}")

## 2. The material database

This is the database the VAE is trained on. Each column is a material attribute. 
The VAE embeds these materials in a continuous latent space.

In [ ]:
mesh, mat_prop, bc, body_force, to_params, vae_params = \
    getMMTOProblemPureStructural(problem)

mat_encoder = MaterialEncoder(vae_params)
mat_encoder.readExcel(xlsx)

tables.materials(mat_encoder, cfg["problem"])

## 3. Latent dimension

The dimension of the latent space is determined using PCA as a heuristic.

PCA on the normalized attributes gives one eigenvalue per direction. The
effective dimension $d_{\mathrm{eff}}$ is the participation ratio of those
eigenvalues, a non-integer count of how many directions carry real variance.
The latent dimension is $n_z = \lceil d_{\mathrm{eff}} \rceil$.

In [ ]:
tables.pca(mat_encoder)

## 4. The problem

Red arrows are the applied load, black markers the fixed and symmetry degrees
of freedom, drawn on the meshed design domain.

In [ ]:
import pyvista as pv
pv.set_jupyter_backend("static")
pv.global_theme.window_size = [1400, 1050]
pv.global_theme.anti_aliasing = "ssaa"
_ = figures.boundary_conditions(problem, size=(920, 675), zoom=0.9)


## 5. The decoder

Set `USE_REFERENCE_VAE = True` to load the VAE used for this benchmark in the
paper. Set it to `False` to train one from scratch.

A freshly trained VAE gives a different latent space, and so a different
design. The run loads whichever network this cell prepares, so the
reconstruction error below is for the decoder it uses.

In [ ]:
import os

USE_REFERENCE_VAE = True

if USE_REFERENCE_VAE:
    net = xlsx.replace(".xlsx", ".nt")        # the network shipped with the database
    mat_encoder.loadAutoencoderFromFile(net)
    print(f"loaded {net}")
else:
    os.makedirs("results", exist_ok=True)
    net = f"results/{EXAMPLE}_trained.nt"
    torch.manual_seed(1234)
    mat_encoder.trainAutoencoder(
        vae_params.numEpochs, vae_params.klFactor, net,
        vae_params.learningRate, vae_params.maxAttributeErrorPercent)
    print(f"trained a fresh network, saved to {net}")

### How faithfully the decoder reproduces the database

Every real material is encoded to its latent point and decoded back. The error reported in the table below
is the difference between the decoded normalized attribute and the real normalized attribute from the database for that material. It measures how much the
embedding distorts the catalogue.

In [ ]:
tables.encoding_errors(mat_encoder)

## 6. How the gradients are computed

The objective, the constraints and the two penalties are computed as one chain of tensor
operations from the design variables, the density $x$ and the latents $z$:

| quantity | depends on |
|---|---|
| $x_{phys}$ | $x$, through the density filter and the Heaviside projection |
| $E$, $\nu$, and $Y$, $\rho$ where the database has them | $z$, through the decoder |
| displacements $\mathbf{u}$ | $x_{phys}$, $E$, $\nu$, through $\mathbf{K}\mathbf{u} = \mathbf{f}$ |
| element stress, von Mises stress $\sigma_{vm}$ | $\mathbf{u}$, $x_{phys}$, $E$, $\nu$ |
| objective $J$ | $\sigma_{vm}$, and $Y$ for the failure factor |
| constraints | $x_{phys}$ and $z$ (volume fraction per material), $\rho$ (mass) |
| penalties | $x_{phys}$, $z$ |

Autograd follows this chain back from each quantity to $x$ and $z$. The one step it cannot
follow by itself is the linear solve. PyTO-AD's sparse solve supplies that step: one adjoint
solve $\mathbf{K}^T \boldsymbol{\lambda} = \partial J / \partial \mathbf{u}$, reusing the
factorization of the forward solve.

The hand-derived sensitivities of `main` (`SensitivitiesPureStructural.py`) are not used on this
branch. The model is `MMTOModelAD` in `1-PureStructural/AutodiffPureStructural.py`.

## 7. Run settings

The settings below are the ones reported in the paper. The gradient check in section 8 and
the run in section 9 both use them.

In [ ]:
run_args = dict(
    to_problem=problem,
    save_dir=f"results/{EXAMPLE}",
    saveNet=net,
    use_pretrained_vae=True,

    relative_filter_radius=cfg["relative_filter_radius"],
    material_filter_radius=cfg["material_filter_radius"],
    pnorm_exponent=cfg.get("pnorm_exponent"),
    maxIterations=cfg["maxIterations"],
    rel_conv_tol=1e-7,

    # distance-to-material penalty
    use_penalization=True,
    gamma_init=1e-3,
    gamma_factor=1.1,
    gamma_max=10,
    penalty_start_iter=0,
    distance_penalty_disable_after_iter=250,
    use_dynamic_gamma_scheduling=True,

    # latent smoothing
    use_z_smoothing=True,
    z_smoothing_weight=1e-4,
    z_smoothing_disable_after_iter=250,

    use_heaviside_projection=True,
    beta_schedule_scheme=cfg["beta"],
    eta_proj=0.5,
    z0_init_method=Z0InitMethod.ORIGIN,
    snap_to_real_material=False,
    use_constant_poissons_ratio=cfg.get("use_constant_poissons_ratio", False),
    constant_poissons_ratio=cfg.get("constant_poissons_ratio", 0.3),
    use_continuation=False,
    binarize_topology=True,
    verbose=True,
)

## 8. Check the gradient

At a random design, the gradient of the objective $J$ and of the first constraint $g_1$ is
compared with central finite differences on a few density and latent variables, for three
step sizes $h$. It takes a minute or two.

The difference has two sources: truncation error, which shrinks with $h$, and round-off in
the solve, which grows as $h$ shrinks. The best agreement is therefore at an intermediate $h$.
A gap that stays the same for every $h$ would mean a wrong gradient.

In [ ]:
gradcheck.check(run_args)

## 9. Run

`run_io.WRITE_FILES = True` saves the result of this run to the `results` folder.

In [ ]:
run_io.WRITE_FILES = False

run_topopt(**run_args)

run = records.from_payload(run_io.LAST_RUN, descriptor=f"my {EXAMPLE} run")
print(run.descriptor)

## 10. The result

In [ ]:
tables.summary(run)

### Optimized design

Coloured by material after binarization.

In [ ]:
_=figures.design(run)

### Latent field

Black stars are the real materials, red points the converged latent coordinates
of the solid elements. Points sitting on a star are elements that resolved onto
a real material; points between them have not.

In [ ]:
_=figures.latent_scatter(run)

### Convergence

The objective on the left axis, each constraint on the right as a fraction of
its budget.

In [ ]:
_=figures.convergence(run)

## 11. Comparison with `main`

The reference runs in `comparison/reference` were made with `comparison/run_both.py`: each
example on `main`, then on `autodiff`, one after the other on one machine, with the settings
of section 7. If you ran section 9 in this session, your run is added as a third column.

First, the settings of the runs, side by side.

In [ ]:
runs = compare.load(EXAMPLE)
if "run" in globals():
    runs["this notebook"] = run

compare.settings(runs)

### Final values and run times

Run time is the optimization alone, as each run recorded it.

In [ ]:
compare.table(runs)

### Histories

The two branches compute the same gradient to six digits or better. MMA amplifies differences
that small over a few hundred iterations, so the histories agree closely at first and drift
apart slowly; the bottom right panel shows by how much.

In [ ]:
_ = compare.histories(runs)

### Designs

Side by side, coloured by material after binarization.

In [ ]:
_ = compare.designs(runs)